## Import de Bibliotecas

In [71]:
import os
import random
import time
import numpy as np

# Função de carga das instâncias

### Transforma os arquivos Knapsack em dicionários, com: 
- capacidade ('c') 
- peso dos objetos ('w') 
- lucro de cada objeto ('p') 
- seleção otimizada de objetos ('s')

In [72]:
def ler_instancia_dicionario(nome: str, pasta: str = f'utils/dados_mochila') -> dict:
    def ler_vetor(sufixo):
        caminho = os.path.join(pasta, nome, f"{nome}_{sufixo}.txt")
        if not os.path.exists(caminho):
            caminho = os.path.join(pasta, f'{nome}_{sufixo}.txt')
        with open(caminho, 'r') as f:
            return np.array([float(x) for x in f.read().strip().split()])

    capacidade = ler_vetor('c')[0]
    pesos = ler_vetor('w')
    lucros = ler_vetor('p')

    otimo_vetor = ler_vetor('s').astype(int)
    lucro_otimo = float(np.sum(lucros * otimo_vetor))

    return {
        'nome': nome,
        'capacidade': capacidade,
        'pesos': pesos,
        'lucros': lucros,
        'n_itens': len(pesos),
        'lucro_otimo': lucro_otimo
    }

# Criação de Indivíduo e População

In [73]:
def criar_individuo(n_itens: int, rng: np.random.Generator) -> np.ndarray:
    return rng.integers(0, 2, size=n_itens)

def criar_populacao(tam_pop: int, n_itens: int, rng: np.random.Generator) -> np.ndarray:
    return rng.integers(0, 2, size=(tam_pop, n_itens))

# Fitness

- **ind * pesos** multiplica posição por posição, então só os itens com 1 contam, e np.sum soma o resultado.
- **peso_total** e **lucro_total** somam o peso e o lucro dos itens que estão na mochila.
- **e_viavel** diz se o peso cabe na capacidade, e serve só para conferir a solução.
- **fitness** é o lucro menos (excesso de peso × maior razão lucro/peso), então estourar a capacidade nunca compensa.
- Exemplo: peso 183, capacidade 165, lucro 233 dão fitness 233 − 18 × 4 = 161.
- Fitness negativo é normal em indivíduos muito inviáveis, e funciona com torneio, mas não com roleta.

    Estrategia: PENALIZACAO.
    fitness = lucro - penalidade * excesso_de_peso

    A penalidade usa a maior razao lucro/peso da instancia, o que garante que
    estourar a capacidade nunca compensa. Se preferir, troque por reparo ou
    fitness zero para inviaveis e compare nos experimentos.

In [74]:
def peso_total(ind, inst):
    return float(np.sum(ind * inst['pesos']))

def lucro_total(ind, inst):
    return float(np.sum(ind * inst['lucros']))

def e_viavel(ind, inst):
    return peso_total(ind, inst) <= inst['capacidade']

def fitness(ind, inst):
    lucro = lucro_total(ind, inst)
    excesso = max(0, peso_total(ind, inst) - inst['capacidade'])
    penalidade = np.max(inst['lucros']/inst['pesos'])
    return lucro - excesso * penalidade

def avaliar_populacao(pop, inst):
    return np.array([fitness(ind, inst) for ind in pop])

# Operadores

In [75]:
def selecao_torneio(pop, fits, k, rng):
    indices = rng.integers(0, len(pop), size=k)
    melhor_indice = indices[np.argmax(fits[indices])]
    return pop[melhor_indice].copy()

def cruzamento(pai1, pai2, taxa_cruzamento, rng):
    numero = rng.random()
    if numero >= taxa_cruzamento:
        return pai1.copy(), pai2.copy()
    ponto = rng.integers(1, len(pai1))
    filho1 = np.concatenate((pai1[:ponto], pai2[ponto:]))
    filho2 = np.concatenate((pai2[:ponto], pai1[ponto:]))
    return filho1, filho2

def mutacao(ind, taxa_mutacao, rng):
    filho = ind.copy()
    mascara = rng.random(len(filho)) < taxa_mutacao
    filho[mascara] = 1 - filho[mascara]
    return filho
        

# Laço Principal

In [76]:
def executar_ag(inst, tam_pop=100, n_geracoes=200, taxa_mutacao=None, taxa_cruzamento=0.9, k_torneio=3, n_elite=2, semente=0, paciencia=None):
    rng = np.random.default_rng(semente)
    n = inst['n_itens']
    if taxa_mutacao is None:
        taxa_mutacao = 1 / n

    inicio = time.time()
    populacao = criar_populacao(tam_pop, n, rng)

    melhor_ind = None
    melhor_fit = -np.inf
    historico = []
    sem_melhora = 0
    elite_qtd = max(0, min(n_elite, tam_pop))

    for i in range(n_geracoes):
        fits = avaliar_populacao(populacao, inst)

        idx = int(np.argmax(fits))
        if fits[idx] > melhor_fit:
            melhor_fit = float(fits[idx])
            melhor_ind = populacao[idx].copy()
            sem_melhora = 0
        else:
            sem_melhora += 1

        historico.append((float(fits[idx]), float(np.mean(fits))))

        if paciencia is not None and sem_melhora >= paciencia:
            break

        nova_pop = []

        if elite_qtd > 0:
            ordem_elite = np.argsort(fits)[::-1][:elite_qtd]
            for i_elite in ordem_elite:
                nova_pop.append(populacao[int(i_elite)].copy())

        while len(nova_pop) < tam_pop:
            pai1 = selecao_torneio(populacao, fits, k_torneio, rng)
            pai2 = selecao_torneio(populacao, fits, k_torneio, rng)

            filho1, filho2 = cruzamento(pai1, pai2, taxa_cruzamento, rng)

            filho1 = mutacao(filho1, taxa_mutacao, rng)
            nova_pop.append(filho1)

            if len(nova_pop) < tam_pop:
                filho2 = mutacao(filho2, taxa_mutacao, rng)
                nova_pop.append(filho2)

        populacao = np.array(nova_pop, dtype=populacao.dtype)

    tempo = time.time() - inicio
    return melhor_ind, melhor_fit, historico, tempo

In [77]:
if __name__ == '__main__':
    inst = [ler_instancia_dicionario(f"p0{i}") for i in range(1, 6)]
    sementes = (0, 1, 2, 3, 4)

    detalhes = []

    print(
        f"{'Inst.':<6} | {'Itens':>5} | {'Cap.':>6} | {'Lucro méd.':>9} | {'Lucro máx.':>9} | "
        f"{'Fit. melhor':>11} | {'Fit. médio':>11} | {'Desvio':>8} | {'Ótimo':>7} | {'Viável':>7} | {'Tempo':>8}"
    )

    for instance in inst:
        lucros = []
        viaveis = 0
        tempos = []
        melhores_inds = []
        melhores_fits = []

        for s in sementes:
            melhor_ind, melhor_fit, historico, tempo = executar_ag(
                instance,
                tam_pop=100,
                n_geracoes=200,
                taxa_mutacao=None,
                taxa_cruzamento=0.9,
                k_torneio=3,
                n_elite=2,
                semente=s,
                paciencia=50
            )
            lucros.append(lucro_total(melhor_ind, instance))
            if e_viavel(melhor_ind, instance):
                viaveis += 1
            tempos.append(tempo)
            melhores_inds.append(melhor_ind)
            melhores_fits.append(melhor_fit)

        melhor_run = int(np.argmax(melhores_fits))
        viavel_pct = 100 * viaveis /len(sementes)

        fitness_melhor = float(np.max(melhores_fits))
        fitness_medio = float(np.mean(melhores_fits))
        fitness_desvio = float(np.std(melhores_fits, ddof=1)) if len(melhores_fits) > 1 else 0.0

        print(
            f"{instance['nome']:<6} | {instance['n_itens']:>5d} | {instance['capacidade']:>6.1f} | "
            f"{np.mean(lucros):>9.1f} | {np.max(lucros):>9.1f} | "
            f"{fitness_melhor:>11.1f} | {fitness_medio:>11.1f} | {fitness_desvio:>8.1f} | "
            f"{instance['lucro_otimo']:>7.1f} | {viavel_pct:>6.1f}% | {np.mean(tempos):>8.4f}"
        )
        detalhes.append((instance, melhores_inds[melhor_run]))


    for instance, ind in detalhes:
        print(
            f"{instance['nome']}: cromossomo={ind} | "
            f"peso={peso_total(ind, instance):.1f}/{instance['capacidade']:.1f} | "
            f"lucro={lucro_total(ind, instance):.1f} (otimo={instance['lucro_otimo']:.1f}) | "
            f"viavel={e_viavel(ind, instance)}"
        )

Inst.  | Itens |   Cap. | Lucro méd. | Lucro máx. | Fit. melhor |  Fit. médio |   Desvio |   Ótimo |  Viável |    Tempo
p01    |    10 |  165.0 |     309.0 |     309.0 |       309.0 |       309.0 |      0.0 |   309.0 |  100.0% |   0.1207
p02    |     5 |   26.0 |      60.0 |      60.0 |        51.6 |        51.6 |      0.0 |    51.0 |    0.0% |   0.0748
p03    |     6 |  190.0 |     164.0 |     164.0 |       159.5 |       159.5 |      0.0 |   150.0 |    0.0% |   0.0734
p04    |     7 |   50.0 |     107.0 |     107.0 |       107.0 |       107.0 |      0.0 |   107.0 |  100.0% |   0.0738
p05    |     8 |  104.0 |    1200.0 |    1200.0 |      1186.0 |      1186.0 |      0.0 |   900.0 |    0.0% |   0.0736
p01: cromossomo=[1 1 1 1 0 1 0 0 0 0] | peso=165.0/165.0 | lucro=309.0 (otimo=309.0) | viavel=True
p02: cromossomo=[1 1 1 0 0] | peso=30.0/26.0 | lucro=60.0 (otimo=51.0) | viavel=False
p03: cromossomo=[1 1 1 0 0 0] | peso=195.0/190.0 | lucro=164.0 (otimo=150.0) | viavel=False
p04: cromosso